# Selecting and Filtering Data

## Introduction

SQL (Structured Query Language) lets you retrieve, filter, and sort data stored in a relational database. This notebook covers the fundamental clauses for reading data from a single table: `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`, `BETWEEN`, and `IS NULL`.

## Objectives

You will be able to:

- Connect to a SQLite database from Python using `sqlite3`
- Write `SELECT` statements to retrieve specific columns
- Filter rows using `WHERE` with single and combined conditions
- Sort results with `ORDER BY` (`ASC` / `DESC`) and cap them with `LIMIT`
- Filter numeric ranges with `BETWEEN` and detect missing values with `IS NULL`
- Wrap query results in a pandas DataFrame

---

## Connecting to a Database

Use `sqlite3.connect()` to open a connection and `.cursor()` to create a cursor object. The cursor tracks where you are in the result set — useful when running multiple queries in sequence.

```python
import sqlite3
conn = sqlite3.connect('data/selecting_data/data.sqlite')
cur = conn.cursor()
```

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('data/selecting_data/data.sqlite')
cur = conn.cursor()

The database we'll use is a CRM (Customer Relationship Management) system for a classic model car company. Here's the schema:

![CRM database schema](assets/selecting_data/Database-Schema.png)

---

## SELECT — Choosing Columns

The `SELECT` statement specifies which columns to retrieve. `*` means all columns.

```sql
SELECT * FROM employees LIMIT 5;
SELECT firstName, lastName, jobTitle FROM employees LIMIT 5;
```

**Running a query:** call `cur.execute()` then `.fetchall()` to get the rows as a list of tuples. Or chain them on one line.

In [ ]:
# Raw tuple output
cur.execute("SELECT * FROM employees LIMIT 3;").fetchall()

**Wrapping in a DataFrame** gives readable column names. Use `cur.description` to get the column names from the last executed query.

In [ ]:
cur.execute("SELECT * FROM employees LIMIT 5;")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

In [ ]:
# Select specific columns only
cur.execute("SELECT firstName, lastName, jobTitle FROM employees LIMIT 5;")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

---

## WHERE — Filtering Rows

The `WHERE` clause restricts which rows are returned. Combine conditions with `AND` / `OR`.

```sql
SELECT * FROM customers WHERE city = 'Boston';
SELECT * FROM customers WHERE city = 'Boston' OR city = 'Madrid';
SELECT * FROM customers WHERE city = 'Madrid' AND creditLimit >= 50000;
```

In [ ]:
cur.execute("SELECT * FROM customers WHERE city = 'Boston';")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

In [ ]:
cur.execute("SELECT * FROM customers WHERE city = 'Boston' OR city = 'Madrid';")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

---

## ORDER BY and LIMIT

`ORDER BY` sorts results by one or more columns. Default is ascending (`ASC`); add `DESC` to reverse. `LIMIT` caps the number of rows returned.

```sql
SELECT customerName, city, creditLimit
FROM customers
ORDER BY creditLimit DESC
LIMIT 10;
```

In [ ]:
# Top 10 customers by credit limit
cur.execute("""
    SELECT customerName, city, creditLimit
    FROM customers
    ORDER BY creditLimit DESC
    LIMIT 10;
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

In [ ]:
# Alphabetical list of unique cities where we have customers
cur.execute("""
    SELECT DISTINCT city
    FROM customers
    ORDER BY city ASC;
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df.head(10)

---

## BETWEEN

`BETWEEN value1 AND value2` is shorthand for `>= value1 AND <= value2`. Both endpoints are inclusive.

```sql
SELECT column_name(s) FROM table_name
WHERE column_name BETWEEN value1 AND value2;
```

In [ ]:
# Customers with a sales rep employee number between 1300 and 1400
cur.execute("""
    SELECT customerName, city, salesRepEmployeeNumber
    FROM customers
    WHERE salesRepEmployeeNumber BETWEEN 1300 AND 1400
    ORDER BY salesRepEmployeeNumber;
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

---

## IS NULL

SQL uses `NULL` for missing values — not zero, not an empty string. The `IS NULL` / `IS NOT NULL` operators test for them.

```sql
SELECT * FROM employees WHERE reportsTo IS NULL;   -- the top of the hierarchy
SELECT * FROM employees WHERE reportsTo IS NOT NULL;
```

In [ ]:
# Which employee has no manager? (top of the org chart)
cur.execute("""
    SELECT firstName, lastName, jobTitle, reportsTo
    FROM employees
    WHERE reportsTo IS NULL;
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

In [ ]:
# Customers with no assigned sales rep
cur.execute("""
    SELECT customerName, city
    FROM customers
    WHERE salesRepEmployeeNumber IS NULL;
""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
print(f"{len(df)} customers have no assigned sales rep")
df.head()

---

## Practice: Famous Dogs

Connect to the dogs database and answer the following queries. The `dogs` table has columns: `name`, `age`, `gender`, `breed`, `temperament`, `hungry`.

| name | age | gender | breed | temperament | hungry |
|------|-----|--------|-------|-------------|--------|
| Snoopy | 3 | M | beagle | friendly | 1 |
| McGruff | 10 | M | bloodhound | aware | 0 |
| Scooby | 6 | M | great dane | hungry | 1 |
| Little Ann | 5 | F | coonhound | loyal | 0 |
| Pickles | 13 | F | black lab | mischievous | 1 |
| Clifford | 4 | M | big red | smiley | 1 |
| Lassie | 7 | F | collie | loving | 1 |
| Snowy | 8 | F | fox terrier | adventurous | 0 |
| NULL | 4 | M | golden retriever | playful | 1 |

In [ ]:
conn2 = sqlite3.connect('data/filtering_and_ordering_lab/dogs.db')
cur2 = conn2.cursor()

In [ ]:
# Select the name and breed for all female dogs


In [ ]:
# Select all dogs listed in alphabetical order by name (note: SQL sorts NULL first)


In [ ]:
# Select any dog that doesn't have a name


In [ ]:
# Select the name and breed of hungry dogs, from youngest to oldest


In [ ]:
# Select the oldest dog's name, age, and temperament


In [ ]:
# Select the three youngest dogs


In [ ]:
# Select the name and breed of dogs between 5 and 10 years old, oldest first


In [ ]:
# Select name, age, and hungry status for hungry dogs aged 2–7, in alphabetical order


---

## Summary

In this notebook you learned how to:

- Connect to a SQLite database and execute queries with `sqlite3`
- Use `SELECT` to choose specific columns and `*` to retrieve all
- Filter rows with `WHERE`, combining conditions using `AND` / `OR`
- Sort results with `ORDER BY ASC/DESC` and cap output with `LIMIT`
- Filter numeric ranges with `BETWEEN` and detect nulls with `IS NULL`
- Wrap query results in a pandas DataFrame with named columns via `cur.description`

Next: [02 — Database Administration](02_database_admin.ipynb)